# 02 · KLASS full sweep

Every item on the KLASS checklist in
[`TESTING_AND_IMPROVEMENTS.md`](../../TESTING_AND_IMPROVEMENTS.md): all 18
`Classification` members, correspondence beyond kommune→fylke, `changes()`
across a real reform, `variants()`, and the ranged `codes(from_=, to=)` path.

In [1]:
import warnings

import pandas as pd

from ssb_statistikkbanken import klass
from ssb_statistikkbanken.klass import Classification

pd.set_option("display.width", 220)
warnings.simplefilter("ignore")

## 1 · Walk every `Classification` member

Previously only `FYLKE` (104) and `KOMMUNE` (131) had ever been fetched.

In [2]:
rows = []
for member in Classification:
    frame = klass.codes(member, date=2025)
    rows.append(
        {
            "name": member.name,
            "id": int(member),
            "rows": len(frame),
            "levels": ", ".join(sorted(frame["level"].dropna().unique())) if len(frame) else "",
        }
    )
sweep = pd.DataFrame(rows)
sweep

,name,id,rows,levels
0,GRUNNKRETS,1,16393,"1, 2"
1,KJONN,2,2,1
2,NARING_SN,6,1785,"1, 2, 3, 4, 5"
3,YRKE_STYRK,7,582,"1, 2, 3, 4"
4,UTDANNING_NUS,36,7253,"1, 2, 3, 4, 5"
5,LANDKODER_SSB3,91,245,1
6,LANDKODER_ALPHA2,100,253,1
7,BYDEL,103,42,1
8,FYLKE,104,16,1
9,HELSEREGION,105,7,1


### Finding: `KOMMUNEKLASSIFISERING` silently returns **0 rows**

Every classification resolves except 129, which comes back empty — no error, no
warning. The cause is that it is a **retired** classification.

In [3]:
klass.default_client().versions(Classification.KOMMUNEKLASSIFISERING)

,id,name,validFrom,validTo
0,470,Kommuneklassifisering 1994-10,1994-10-01,2019-01-01


It was only valid **1994-10-01 → 2019-01-01**. Asking for 2025 is outside that
window, and KLASS answers with an empty `codes` list rather than an error.

Inside the window it works fine:

In [4]:
valid = klass.codes(Classification.KOMMUNEKLASSIFISERING, date=2018)
print(f"date=2018 -> {len(valid)} rows")
valid[["code", "name"]]

date=2018 -> 7 rows


,code,name
0,1,Primærnæringskommuner
1,2,Blandede landbruks- og industrikommuner
2,3,Industrikommuner
3,4,"Mindre sentrale, blandede tjenesteytings- og i..."
4,5,"Sentrale, blandede tjenesteytings- og industri..."
5,6,Mindre sentrale tjenesteytingskommuner
6,7,Sentrale tjenesteytingskommuner


The dangerous form is `mapping()`, because an empty dict is a perfectly valid
dict — every downstream `.map()` would silently produce `NaN` with no clue why.

**Fixed (F-01):** the wrapper now warns on a zero-code answer and names the
classification's real validity window, so the date is actually fixable.

In [5]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    retired = klass.mapping(Classification.KOMMUNEKLASSIFISERING, date=2025)

print(f"mapping(129, date=2025) -> {type(retired).__name__} with {len(retired)} entries")
for warning in caught:
    print(f"\n{warning.category.__name__}: {warning.message}")

mapping(129, date=2025) -> dict with 0 entries



## 2 · `correspondence()` beyond kommune→fylke

In [6]:
econ = klass.correspondence(Classification.KOMMUNE, Classification.OKONOMISK_REGION, date=2025)
print(f"KOMMUNE -> OKONOMISK_REGION: {len(econ)} rows")
econ.head()

KOMMUNE -> OKONOMISK_REGION: 357 rows


,sourceCode,sourceName,targetCode,targetName
0,0301,Oslo,03001,Oslo
1,1101,Eigersund,11001,Dalane
2,1103,Stavanger,11002,Stavanger/Sandnes
3,1106,Haugesund,11005,Haugaland
4,1108,Sandnes,11002,Stavanger/Sandnes


A pair that has **no** correspondence table gives an excellent error — KLASS's
own prose, passed straight through:

In [7]:
from ssb_statistikkbanken import KlassError

try:
    klass.correspondence(Classification.KOMMUNE, Classification.BYDEL, date=2025)
except KlassError as exc:
    print(f"{type(exc).__name__}:\n{exc}")

ClassificationNotFoundError:
KLASS has no /classifications/131/correspondsAt (HTTP 404). Classification 'Standard for kommuneinndeling' has no correspondence table with Classification 'Standard for bydelsinndeling'


## 3 · `changes()` across the 2020 kommune reform

The January 2020 reform merged a large number of municipalities — the single
best real-world test of this endpoint.

In [8]:
reform = klass.changes(Classification.KOMMUNE, from_=2019, to=2021)
print(f"{len(reform)} change records")
merged_into_stavanger = reform[reform["newName"] == "Stavanger"]
merged_into_stavanger

317 change records


,oldCode,oldName,newCode,newName,changeOccurred
0,1103,Stavanger,1103,Stavanger,2020-01-01
1,1141,Finnøy,1103,Stavanger,2020-01-01
2,1142,Rennesøy,1103,Stavanger,2020-01-01


Finnøy (1141) and Rennesøy (1142) folding into Stavanger (1103) is exactly what
happened in reality, so the endpoint and the parsing are both correct.

## 4 · `variants()`

In [9]:
for member in [
    Classification.NARING_SN,
    Classification.UTDANNING_NUS,
    Classification.YRKE_STYRK,
    Classification.FYLKE,
]:
    found = klass.default_client().variants(member, date=2025)
    print(f"{member.name:<16} {len(found):>2} variant(s)")
    for name in found["name"].head(2):
        print(f"    - {name}")

NARING_SN         1 variant(s)
    - Arbeidsmarkeds- og lønnsstatistikk (17 kategorier) 2025  - variant av Næringsgruppering (SN) 2025


UTDANNING_NUS    16 variant(s)
    - Studiepoeng ved fagskole 2024-09  - variant av Utdanningsgruppering (NUS) 2024-09
    - Fagskoleutdanning 2024-09  - variant av Utdanningsgruppering (NUS) 2024-09


YRKE_STYRK        4 variant(s)
    - Yrkesklassifisering for oppdragstakere 2019-04  - variant av Yrkesklassifisering 08 (STYRK-08) 
    - Yrkesgruppering - lønnstakere med høyere utdanning 2011  - variant av Yrkesklassifisering 08 (STYRK-08) 


FYLKE             2 variant(s)
    - Produksjonsfylke 2024  - variant av Fylkesinndeling 2024
    - bokmål navn 2024  - variant av Fylkesinndeling 2024


## 5 · Ranged `codes(from_=, to=)` vs the `codesAt` snapshot

The ranged form returns codes valid at **any point** in the window, so it picks
up the pre-2020 counties that a 2025 snapshot cannot see.

In [10]:
snapshot = klass.codes(Classification.FYLKE, date=2025)
ranged = klass.codes(Classification.FYLKE, from_=2019, to=2021)
print(f"snapshot date=2025      : {len(snapshot)} codes")
print(f"ranged   from_=2019,to=2021: {len(ranged)} codes")

gone = set(ranged["code"]) - set(snapshot["code"])
print(f"\ncodes only visible in the ranged view: {len(gone)}")
ranged[ranged["code"].isin(sorted(gone))][["code", "name", "validFrom", "validTo"]].head(12)

snapshot date=2025      : 16 codes
ranged   from_=2019,to=2021: 27 codes

codes only visible in the ranged view: 16


,code,name,validFrom,validTo
0,01,Østfold,None,None
1,02,Akershus,None,None
3,04,Hedmark,None,None
4,05,Oppland,None,None
5,06,Buskerud,None,None
6,07,Vestfold,None,None
7,08,Telemark,None,None
8,09,Aust-Agder,None,None
9,10,Vest-Agder,None,None
11,12,Hordaland,None,None


Worth noting: `validFrom` / `validTo` come back as `None` even in ranged mode,
so the frame tells you *which* codes existed in the window but not *when* each
one stopped being valid. That is KLASS's response shape, not a wrapper bug.